# 🥇 Gold Layer — Incremental Processing

> **Incremental gold table builds** with **latest-state** and **timestamp-based volume snapshots**.

This notebook performs the following:
1. **Incremental merge** into gold-level tables from silver sources
2. **Latest snapshot** — a deduplicated view of the most current records
3. **Timestamp volume snapshots** — point-in-time record counts for trend tracking

## 📦 Step 1: Import & Setup

This step handles all initial configuration:

- **Import** the required helper modules
- **Switch** to the correct catalog and ensure the `gold` schema exists (or create it)
- **Generate** a unique `gold_run_id`
- **Capture** the run date and run timestamp as strings

> 💡 These values are used for **tracking** and **snapshot publishing** throughout the pipeline.

In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog novacartdbx")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_schema")
gold_run_id = str(uuid.uuid4())

run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print("Current Gold Run id:", gold_run_id)
print("Run Timestamp folder:", run_ts_str)

   
## 🗂️ Step 2: Create Control Table

The **control table** stores the latest gold processing state. It tells the gold layer:

- **Which silver data** was processed in the last run
- **How many records** were merged during the last gold run

> 💡 This enables **incremental processing** by tracking exactly where the previous run left off.

In [0]:
spark.sql("""
          create table if not exists novacartdbx.gold_schema.processing_control
          (
            layer string,
            enity_name string,
            last_process_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged string,
            run_status string,
            gold_run_id string,
            updated_at timestamp
          )
          using delta
          """)

   
## 🛠️ Step 3: Helper Functions

Three utility functions power the incremental gold merge workflow:

| Function | Purpose |
|---|---|
| **`upsert_to_gold`** | Merges incoming data into the current gold state table (insert or update) |
| **`get_last_processed_silver_ts`** | Reads the gold watermark from the control table to determine where to resume |
| **`upsert_gold_control`** | Updates the gold control table after a successful run |

> 💡 Together, these functions ensure **idempotent**, **incremental** processing from silver → gold.

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target").merge(
            df_source.alias("source"), f"target.{join_key} = source.{join_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name: str):
    ctrl = (
        spark.table("novacartdbx.gold_schema.processing_control")
        .filter(
            (f.col("layer") == 'gold')&
            (f.col("enity_name") == entity_name)&
            (f.col("run_status") == 'SUCCESS')
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None
    else:
        return rows[0]["last_processed_silver_run_ts"]
      

In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_silver_run_ts,rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_silver_run_ts,
            str(int(rows_merged)),
            "SUCCESS",
            gold_run_id,
            datetime.now()
        )],
        schema = """
        layer string,
        enity_name string,
        last_process_silver_run_id string,
        last_processed_silver_run_ts timestamp,
        rows_merged string,
        run_status string,
        gold_run_id string,
        updated_at timestamp
        """  
        )
    dt = DeltaTable.forName(spark,"novacartdbx.gold_schema.processing_control")
    (dt.alias("t").merge(
        ctrl_df.alias("s"), f"t.layer = s.layer and t.enity_name = s.enity_name")
    .whenMatchedUpdate(
        set={
            "last_process_silver_run_id": "s.last_process_silver_run_id",
            "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "gold_run_id": "s.gold_run_id",
            "updated_at": "s.updated_at"
        }
    )
    .whenNotMatchedInsertAll()
    .execute())

   
   
## 🔄 Step 4: Read Changed Silver Rows Only

Read the full current silver state table, but **filter out only the rows that changed** since the last gold run.

This is the **starting point** of gold incremental processing:

- 🕒 **Retrieve** the last processed silver timestamp from the control table
- 📥 **Read** the silver table and keep only rows with a newer `silver_run_ts`
- ⚡ **Minimize** data movement by processing only what's new or updated

> 💡 This ensures the gold layer stays **efficient** and **incremental** — no full reprocessing needed.

In [0]:
last_gold_ts = get_last_processed_silver_ts("order_information")

print("Last Processed Silver timestamp for gold:", last_gold_ts)

silver_order_current = spark.read.table("novacartdbx.silver_schema.orders_transformed")
silver_product_current = spark.read.table("novacartdbx.silver_schema.products_transformed")
silver_payment_current = spark.read.table("novacartdbx.silver_schema.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_order_current
    changed_products = silver_product_current
    changed_payments = silver_payment_current
else:
    changed_orders = silver_order_current.filter(f.col("updated_at") > f.lit(last_gold_ts))
    changed_products = silver_product_current.filter(f.col("updated_at") > f.lit(last_gold_ts))
    changed_payments = silver_payment_current.filter(f.col("processed_at") > f.lit(last_gold_ts))

changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"Number of changed orders: {changed_orders_count}")
print(f"Number of changed products: {changed_products_count}")
print(f"Number of changed payments: {changed_payments_count}")


   
   
## 🎯 Step 5: Identify Impacted Order IDs

The gold layer is built at the **order grain**, so any change in **orders**, **products**, or **payments** must be traced back to its parent `order_id`.

- 🔍 **Detect** which `order_id`s were touched by changed silver rows
- 🧩 **Union** impacted IDs from all three sources into a single distinct set
- ♻️ **Rebuild** only those orders in the gold table — no full reprocessing

> 💡 This keeps the gold refresh **surgical** and **efficient** — only affected orders are recomputed.

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p")
    .join(silver_order_current.alias("o"),
          f.col("p.product_id") == f.col("o.product_id"),"inner")
    .select(f.col("o.order_id")).distinct()
)

impacted_order_ids = (
    impacted_from_orders.union(impacted_from_orders)
    .union(impacted_from_products)
    .distinct()
)

print(f"impacted_orders_id: ", impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))


In [0]:
impacted_orders = (
    silver_order_current.alias("o")
    .join(impacted_order_ids.alias("i"),
          f.col("o.order_id") == f.col("i.order_id"),"inner")
)

gold_delta = (
    impacted_orders
    .join(silver_product_current.alias("p"),
          f.col("o.product_id") == f.col("p.product_id"),"inner")
    .join(
        silver_payment_current.alias("pay"),
        f.col("o.order_id") == f.col("pay.order_id"),"inner")
    .select(
        f.col("o.order_id"),
        f.col("o.customer_id"),
        f.col("p.product_id"),
        f.col("p.product_name"),
        f.col("p.category"),
        f.col("p.price").alias("product_price"),
        f.col("o.order_status"),
        f.col("o.order_amount"),
        f.col("pay.payment_id"),
        f.col("pay.payment_status"),
        f.col("pay.paid_amount"),
        f.col("o.order_date"),
        f.col("o.order_month"),
        f.col("order_year"),
        f.greatest(
            f.col("o.updated_at").cast("timestamp"),
            f.col("p.updated_at").cast("timestamp"),
            f.col("pay.processed_at").cast("timestamp")
        ).alias("gold_updated_ts"),
        )
    .dropDuplicates(["order_id"])
    .withColumn("payment_completion_ratio",
                f.when(
                    f.col("order_amount") > 0,
                    f.col("paid_amount") / f.col("order_amount")
                )
                .otherwise(f.lit(0.0))
                )
    .withColumn(
        "payment_state",
        f.when(f.col("order_amount") == 0, "Invalid_order_amount")
        .when(f.col("payment_completion_ratio") == 0, "unpaid")
        .when(f.col("payment_completion_ratio") == 1, "paid")
        .when(f.col("payment_completion_ratio") > 0, "Overpaid")
        .when(f.col("payment_completion_ratio") < 1, "partially_paid")   
    )
    .withColumn("gold_upadted_date", f.to_date(f.col("gold_updated_ts")))
    .withColumn("gold_run_id",f.lit(gold_run_id))
)

print("gold_delta_rows = ", gold_delta.count())
display(gold_delta)


   
   
## 🔀 Step 6: Merge into Gold Current-State Table

If the gold delta contains new or updated rows, this step **merges** them into `gold_schema.orders_information`.

- ✅ **Rows found** → upsert (insert new / update existing) into the gold table
- ⏭️ **No rows** → skip the merge entirely, nothing is written

> 💡 This ensures the gold table always reflects the **latest state** of each order — efficiently and incrementally.

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "novacartdbx.gold_schema.orders_information", "order_id")
else:
    print("No new rows to insert in gold table")    

In [0]:
%sql
select * from novacartdbx.gold_schema.orders_information

   
   
   
## 🕰️ Step 7: Maintain Gold SCD Type 2 History

When a current gold row **changes**, this step preserves the full history using **Slowly Changing Dimension Type 2** logic:

- 🔒 **Close** the old version → set `is_current = false` and stamp the `valid_to` timestamp
- ➕ **Insert** the new version → set `is_current = true` with an open `valid_to`

> 💡 This ensures the gold layer maintains a **complete audit trail** of every order state change over time.

In [0]:
if not spark.catalog.tableExists("novacartdbx.gold_schema.orders_information_scd2"):
    spark.sql("""
              create table novacartdbx.gold_schema.orders_information_scd2
              using delta
              as
              select *,
              cast(null as timestamp) as valid_from_ts,
              cast(null as timestamp) as valid_to_ts,
              true as is_current from novacartdbx.gold_schema.orders_information
              where 1 = 0
              """)
    
if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
              merge into novacartdbx.gold_schema.orders_information_scd2 t
              using gold_delta_view s
              on t.order_id = s.order_id and t.is_current = true
              when matched and (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price)
              )
              then update set
                  is_current = false,
                  valid_to_ts = s.gold_updated_ts
              """)
    
    
    spark.sql("""
              insert into novacartdbx.gold_schema.orders_information_scd2
              select s.*,
              s.gold_updated_ts as valid_from_ts,
              cast(null as timestamp) as valid_to_ts,
              true as is_current
              from gold_delta_view s 
              left join novacartdbx.gold_schema.orders_information_scd2 t
              on s.order_id = t.order_id and t.is_current = true
              where t.order_id is null or (
                  not(t.order_status <=> s.order_status) or
                not(t.order_amount <=> s.order_amount) or
                not(t.paid_amount <=> s.paid_amount) or
                not(t.payment_id <=> s.payment_id) or
                not(t.category <=> s.category) or
                not(t.product_name <=> s.product_name) or
                not(t.product_price <=> s.product_price)
              )
              
              """)

   
   
   
## 📊 Step 8: Update Category-Level Gold Aggregation

Recalculate category-level business metrics **only for categories impacted** in the current run, then merge them into the category performance gold table.

- 🔍 **Identify** which product categories were touched by the current batch
- 📈 **Recompute** aggregated KPIs (revenue, order count, avg order value, etc.) for those categories only
- 🔀 **Merge** the refreshed metrics into `gold_schema.category_performance`

> 💡 This keeps category-level rollups **accurate** and **efficient** — only affected categories are recalculated, not the entire table.

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta.select("category")
        .filter(f.col("category").isNotNull())
        .distinct()
    )

    category_perf_delta = (
        spark.read.table("novacartdbx.gold_schema.orders_information")
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            f.countDistinct("order_id").alias("total_orders"),
            f.sum(
                f.when(f.col("order_amount") > 0, f.col("order_amount"))
                .otherwise(f.lit(0.0))
            ).alias("Gross_Merchandise_Value"),
            f.sum(
                f.when(f.col("paid_amount") > 0, f.col("paid_amount"))
                .otherwise(f.lit(0.0))
            ).alias("Total_paid_amount"),
            f.avg(f.col("payment_completion_ratio")).alias("avg_payment_completion_ratio"),
            (
                f.sum(f.when(f.col("payment_status") == "FAILED", 1).otherwise(0)) / f.count("*")
            ).alias("payment_Failure_rate")
        )
    )

upsert_to_gold(category_perf_delta, "novacartdbx.gold_schema.category_performance", "category")    

   
   
   
## 📸 Step 9: Publish Gold Snapshot to Volume

Write two types of gold output to a **Databricks Volume** for portability and external access:

| Snapshot Type | Description |
|---|---|
| **Latest Snapshot** | Overwritten on every successful run — always reflects the current gold state |
| **Timestamped Historical Snapshot** | A new folder for each successful run — preserves every version |

> 💡 This is useful for **audit trails**, **rollback scenarios**, and **teaching demos** where you need reproducible, point-in-time gold data outside Delta tables.


In [0]:
spark.sql("create volume if not exists novacartdbx.gold_schema.gold_schema_vol")

In [0]:
latest_orders_path = (
    "/Volumes/novacartdbx/gold_schema/gold_schema_vol/gold_latest/orders_information"
)

latest_category_path = (
    "/Volumes/novacartdbx/gold_schema/gold_schema_vol/gold_latest/category_performace"

)


historical_orders_path = f"/Volumes/novacartdbx/gold_schema/gold_schema_vol/gold_snapshots/order_information/run_date={run_date_str}/run_ts={run_ts_str}"


historical_category_path = f"/Volumes/novacartdbx/gold_schema/gold_schema_vol/gold_snapshots/category_performace/run_date={run_date_str}/run_ts={run_ts_str}"


spark.read.table("novacartdbx.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)
spark.read.table("novacartdbx.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)
spark.read.table("novacartdbx.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table("novacartdbx.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)


print("latest orders path: ", latest_orders_path)
print("latest category path: ", latest_category_path)

print("Historical order path: ", historical_orders_path)
print("Historical category path: ", historical_category_path)

   
   
   
   
## ✅ Step 10: Update Gold Control Table

After a successful gold run, **record the processing metadata** in the gold control table for auditability and incremental tracking.

- 📝 **Log** the latest silver run ID and timestamp that was processed
- 📊 **Record** the number of rows merged and the run status
- 🔍 **Display** the control table for validation and monitoring

> 💡 This closes the loop on the gold pipeline — ensuring the next run knows **exactly where to resume** from.

In [0]:
latest_silver_ts = silver_order_current.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

latest_silver_run_id = (
    silver_order_current.filter(f.col("bronze_ingested_at") == latest_silver_ts)
    .agg(f.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None


upsert_gold_control("order_information", latest_silver_run_id,latest_silver_ts,gold_delta.count())

display(spark.table("novacartdbx.gold_schema.processing_control"))

In [0]:
%sql
select * from novacartdbx.gold_schema.processing_control;